In [2]:
import torch
!rocminfo | head
print("-----------------------------------------------")
print("PyTorch:", torch.__version__)
print("HIP version:", torch.version.hip)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device count:", torch.cuda.device_count())
    print("Device:", torch.cuda.get_device_name(0))
print("-----------------------------------------------")
!rocminfo | grep -E "Name:|Marketing Name:"

ROCk module is loaded
HSA System Attributes    
Runtime Version:         1.18
Runtime Ext Version:     1.14
System Timestamp Freq.:  1000.000000MHz
Sig. Max Wait Duration:  18446744073709551615 (0xFFFFFFFFFFFFFFFF) (timestamp count)
Machine Model:           LARGE                              
System Endianness:       LITTLE                             
-----------------------------------------------
PyTorch: 2.9.1+rocm7.1.1.git351ff442
HIP version: 7.1.52802-26aae437f6
CUDA available: True
Device count: 1
Device: AMD Radeon RX 9070 XT
-----------------------------------------------
  Name:                    AMD Ryzen 9 5900XT 16-Core Processor
  Marketing Name:          AMD Ryzen 9 5900XT 16-Core Processor
  Vendor Name:             CPU                                
  Name:                    gfx1201                            
  Marketing Name:          AMD Radeon RX 9070 XT              
  Vendor Name:             AMD                                
      Name:                    

# Pour LSTM

In [ ]:
import torch
from models.ParamGrid import ParamGrid, Experiment, DirectConstructor
from models.acer_conf_lstm import AcerConfLSTM, MLPBlock
import models.activations as act_f
from models.training import TrainingArgs, Exp_Args, exp_acer_data_module
from classes.ACER_data_handler import ACERDataModule
import operator
import utils.yaml.YAML_register_groups as YAMLGroups

YAMLGroups.register_torch()

LSTMTester = ParamGrid(
    param_grid={
        "shared_params": {
            "lookback_days": [7, 14, 28],
            "horizon": [1, 3, 7],
        },
        "model_LSTM": {
            "lstm_input_size": [25],
            "lstm_hidden_size": [128, 256, 512, 768],
            "lstm_num_layers": [1,2, 3],
            "lstm_dropout": [0.0, 0.1, 0.2, 0.3, 0.5],
            "lstm_lookback": [1],
            "temperature_forecast": [True],
            "mlp_config": [
                None,
                [
                    MLPBlock.SequentialBlock(92, layer_norm=True, activation=act_f.SiLU(), dropout_rate=0.2)
                ],
                [
                    MLPBlock.SequentialBlock(192, layer_norm=True, activation=act_f.SiLU(), dropout_rate=0.2),
                    MLPBlock.SequentialBlock(92, layer_norm=True, activation=act_f.SiLU(), dropout_rate=0.2)
                ],
            ],
        },
        "data_module_params": {
            "start_padding_days": [2],
            "end_padding_days": [0],
            "norm_by_dayly_deviation": [True],
            "sample_hour": [16],
            "augment_data": [False],
            "year_no_augment": [[2017, 2024]],
            "val_split": [[2017, 2024]],
            "normalize_volume": [False],
            "normalize_temperature": [True],
            "scaling_mode_robust": [True],
            "scaling_per_feature": [False],
            "return_fields": [
                [
                    f"concat[input_temperature_lagged_view[-1,${{n_lookback_days}},24], input_volume_lagged_view[-1,${{n_lookback_days}},1]]",
                    "input_temperature_forecast",
                    "target_volume",
                    "volume_journalier_moyen",
                ]
            ],
        },
        "training_args": {
            "num_epochs": [500],
            "batch_size": [64],
            "learning_rate": [0.0001],
            "lstm_lr": [0.00005],
            "weight_decay": [1e-4],
            "dataType": [torch.bfloat16],
            "verbose": [False],
            "early_stopping_patience": [75],
            "early_stopping_min_delta": [0.00001],
            "early_stopping_min_epoch": [100],
        },
    },
)

new_lstm_exp = Experiment(
    comment="LSTM explorer plus de dropout - SiLU only",
    folder_path="models_testing/LSTM/EXP_dropout_explorer",
    param_grid=LSTMTester,
    experiment_function=exp_acer_data_module,
    exp_args_mapping={
        "model_ExpConstructor": Experiment.ExpArgConstructor(
            constructor = DirectConstructor,
            exp_args_mapping = {
                "constructor$&noSearch" : AcerConfLSTM,
                "args" : {
                     "__$&UnpackDirect": "model_LSTM",
                    "horizon" : "shared_params.horizon"
                }
            }
        ),
        "data_module": Experiment.ExpArgConstructor(
            constructor = ACERDataModule,
            exp_args_mapping = {
                "__$&UnpackDirect": "data_module_params",
                "n_lookback_days" : "shared_params.lookback_days",
                "horizon" : "shared_params.horizon"
            }
        ),
        "training_args" : Experiment.ExpArgConstructor(
            constructor = TrainingArgs,
            exp_args_mapping = {
                "__$&UnpackDirect": "training_args",
            }
        ),
        "exp_args": Experiment.ExpArgConstructor(
            constructor = Exp_Args,
            exp_args_mapping = {
                "path" : "__$&FolderPath",
                "model_base_folder$&noSearch" : "__$&ExpPath/saved_models/",
                "model_folder_division" : {
                    "horizon" : "shared_params.horizon",
                    "lookback_days" : "shared_params.lookback_days"
                },
                "nb_trials$&noSearch" : 2,
                "exp_number" : "__$&RunIndex"
            }
        )
    }
)

# Le nombre de combinaisons
print(f"Nombre de combinaisons : {new_lstm_exp.get_combinations_count()}")

new_lstm_exp.save_to_yaml()

Nombre de combinaisons : 1620


# Pour MeteoMixer

In [6]:
import torch
from models.ParamGrid import ParamGrid, Experiment, DirectConstructor
from models.CustomMixer import MeteoEmbConfig, MeteoTSMixer, MeteoTSMixer_withDailyVar
from models.TSMixer import Detailled_hidden_dim_Config as TSM_HidDim_Conf, TSMixerExt_Config, NormPlacement
from models.training import TrainingArgs, Exp_Args, exp_acer_data_module
from classes.ACER_data_handler import ACERDataModule
import operator
import utils.yaml.YAML_register_groups as YAMLGroups

YAMLGroups.register_torch()

newTester = ParamGrid(
    param_grid={
        "shared_params": {
            "lookback_days": [7, 14, 28],
            "horizon": [1],
            "meteo_length": [24],
            "nb_daily_features": [1],
            "nb_daily": [2],
            "nb_predic_features": [1]
        },
        "model_MeteoEmbConfig": {
            "feature_dim" : [2],
            "conv_dim" : [32, 64],
            "num_conv_layers" : [3],
            "attention_dim" : [32],
            "num_attention_heads" : [4],
            "output_dim" : [6,12,24]
        },
        "model_TSMixerExt": {
            "mixing_hidden_dim" : [64],
            "k_mixing_layers" : [3],
            "norm_placement" : [NormPlacement.POST],
            "activation" : [torch.nn.SiLU()],
            "dropout_rate" : [0.15]
        },
        "data_module_params": {
            "start_padding_days": [2],
            "end_padding_days": [0],
            "norm_by_dayly_deviation": [True],
            "sample_hour": [16],
            "augment_data": [False],
            "year_no_augment": [[2017, 2024]],
            "val_split": [[2017, 2024]],
            "normalize_volume": [False],
            "normalize_temperature": [True],
            "scaling_mode_robust": [True],
            "scaling_per_feature": [False],
            "return_fields": [
                [
                    f"concat[" #Hourly features of the past n_lookback_days
                        f"input_temperature_lagged_view[-1,${{n_lookback_days}},24,1], "
                        f"weather_lagged(shortwave_radiation)_view[-1,${{n_lookback_days}},24,1]"
                        # f"weather_lagged(precipitation)_view[-1,${{n_lookback_days}},24,1], "
                        # f"weather_lagged(cloud_cover)_view[-1,${{n_lookback_days}},24,1], "
                        # f"weather_lagged(wind_speed_10m)_view[-1,${{n_lookback_days}},24,1], "
                        # f"weather_lagged(vapour_pressure_deficit)_view[-1,${{n_lookback_days}},24,1]"
                        f"]",
                    f"concat[dj_0_sum_lagged_view[-1,${{n_lookback_days}},1], input_volume_lagged_view[-1,${{n_lookback_days}},1]]",
                    f"concat[" #Hourly features of the future n_horizon_days
                        f"input_temperature_forecast_view[-1,${{horizon}},24,1], "
                        f"weather_forecast(shortwave_radiation)_view[-1,${{horizon}},24,1]"
                        # f"weather_forecast(precipitation)_view[-1,${{horizon}},24,1], "
                        # f"weather_forecast(cloud_cover)_view[-1,${{horizon}},24,1], "
                        # f"weather_forecast(wind_speed_10m)_view[-1,${{horizon}},24,1], "
                        # f"weather_forecast(vapour_pressure_deficit)_view[-1,${{horizon}},24,1]"
                        f"]",
                    f"dj_0_sum_forecast_view[-1,${{horizon}},1]",
                    f"target_volume_view[-1,${{horizon}},1]",
                    "volume_journalier_moyen",
                ]
            ],
        },
        "training_args": {
            "num_epochs": [1000],
            "batch_size": [32],
            "learning_rate": [0.005],
            "lstm_lr": [0.01,],
            "weight_decay": [1e-4],
            "dataType": [torch.bfloat16],
            "verbose": [False],
            "early_stopping_patience": [75],
            "early_stopping_min_delta": [0.000001],
            "early_stopping_min_epoch": [100],
        },
    },
)

new_exp = Experiment(
    comment="MeteoTSMixer_withDailyVar H1 - Simple temperature, radiation solaire et de la somme des degres jours DJ0",
    folder_path="models_testing/MeteoTSMixer_withDailyVar/EXP_2_0",
    param_grid=newTester,
    experiment_function=exp_acer_data_module,
    exp_args_mapping={
        "model_ExpConstructor": Experiment.ExpArgConstructor(
            constructor = DirectConstructor,
            exp_args_mapping = {
                "constructor$&noSearch" : MeteoTSMixer_withDailyVar,
                "args" : {
                    "MeteoEmbConfig" : Experiment.ExpArgConstructor(
                        constructor = MeteoEmbConfig,
                        exp_args_mapping = {
                            "__$&UnpackDirect" : "model_MeteoEmbConfig"
                        }
                    ),
                    "TSMixerConfig" : Experiment.ExpArgConstructor(
                        constructor = TSMixerExt_Config,
                        exp_args_mapping = {
                            "historical_time_dim" : "shared_params.lookback_days",
                            "historical_channel_dim" : Experiment.ExpArgConstructor(
                                constructor = operator.add,
                                exp_args_mapping = {
                                    "a" : "model_MeteoEmbConfig.output_dim",
                                    "b" : "shared_params.nb_daily"
                                }
                            ),
                            "future_time_dim" : "shared_params.horizon",
                            "future_channel_dim" : Experiment.ExpArgConstructor(
                                constructor = operator.add,
                                exp_args_mapping = {
                                    "a" : "model_MeteoEmbConfig.output_dim",
                                    "b" : "shared_params.nb_daily_features"
                                }
                            ),
                            "mixing_hidden_dim" : "model_TSMixerExt.mixing_hidden_dim",
                            "k_mixing_layers" : "model_TSMixerExt.k_mixing_layers",
                            "output_channel_dim" : "shared_params.nb_predic_features",
                            "norm_placement" : "model_TSMixerExt.norm_placement",
                            "activation" : "model_TSMixerExt.activation",
                            "dropout_rate" : "model_TSMixerExt.dropout_rate",
                        }
                    ),
                }
            }
        ),
        "data_module": Experiment.ExpArgConstructor(
            constructor = ACERDataModule,
            exp_args_mapping = {
                "__$&UnpackDirect": "data_module_params",
                "n_lookback_days" : "shared_params.lookback_days",
                "horizon" : "shared_params.horizon"
            }
        ),
        "training_args" : Experiment.ExpArgConstructor(
            constructor = TrainingArgs,
            exp_args_mapping = {
                "__$&UnpackDirect": "training_args",
            }
        ),
        "exp_args": Experiment.ExpArgConstructor(
            constructor = Exp_Args,
            exp_args_mapping = {
                "path" : "__$&FolderPath",
                "model_base_folder$&noSearch" : "__$&ExpPath/saved_models/",
                "model_folder_division" : {
                    "dType" : "training_args.dataType",
                    "horizon" : "shared_params.horizon",
                    "lookback_days" : "shared_params.lookback_days"
                },
                "nb_trials$&noSearch" : 2,
                "exp_number" : "__$&RunIndex"
            }
        )
    }
)

# Le nombre de combinaisons
print(f"Nombre de combinaisons : {new_exp.get_combinations_count()}")

new_exp.save_to_yaml()

Nombre de combinaisons : 18


# Pour TSMixerExt

In [8]:
import torch
from models.ParamGrid import ParamGrid, Experiment, DirectConstructor
from models.TSMixer import TSMixerExt_PositiveOutput, Detailled_hidden_dim_Config as TSM_HidDim_Conf, TSMixerExt_Config, NormPlacement
from models.training import TrainingArgs, Exp_Args, exp_acer_data_module
from classes.ACER_data_handler import ACERDataModule
import operator
import utils.yaml.YAML_register_groups as YAMLGroups

YAMLGroups.register_torch()

nb_meteo_features = 1

newTester = ParamGrid(
    param_grid={
        "shared_params": {
            "lookback_days": [7, 14, 28],
            "horizon": [1, 3, 7],
            "meteo_length": [24],
            "nb_meteo_features": [nb_meteo_features],
            "nb_predic_features": [1]
        },
        "model_TSMixerExt": {
            "historical_channel_dim" : [nb_meteo_features*24 + 1],  # 2 meteo features + 1 degree jour + 1 volume feature
            "future_channel_dim" : [nb_meteo_features*24],  # 2 meteo features + 1 degree jour
            "mixing_hidden_dim" : [48,64,96,128],
            "k_mixing_layers" : [2,3,4],
            "norm_placement" : [NormPlacement.POST],
            "activation" : [torch.nn.SiLU()],
            "dropout_rate" : [0, 0.15, 0.3]
        },
        "data_module_params": {
            "start_padding_days": [2],
            "end_padding_days": [0],
            "norm_by_dayly_deviation": [True],
            "sample_hour": [16],
            "augment_data": [False],
            "year_no_augment": [[2017, 2024]],
            "val_split": [[2017, 2024]],
            "normalize_volume": [False],
            "normalize_temperature": [True],
            "scaling_mode_robust": [True],
            "scaling_per_feature": [False],
            "return_fields": [
                [
                    f"concat["
                        f"input_temperature_lagged_view[-1,${{n_lookback_days}},24], "
                        # f"weather_lagged(apparent_temperature)_view[-1,${{n_lookback_days}},24], "
                        # f"weather_lagged(shortwave_radiation)_view[-1,${{n_lookback_days}},24], "
                        # f"regex:^T-[0-9]+_(temp_min)$::_view[-1,${{n_lookback_days}},1], "
                        # f"regex:^T-[0-9]+_(temp_max)$::_view[-1,${{n_lookback_days}},1], "
                        # f"regex:^T-[0-9]+_(dj_0)$::_view[-1,${{n_lookback_days}},1], "
                        # f"regex:^T-[0-9]+_(dj_0_sum)$::_view[-1,${{n_lookback_days}},1], "
                        # f"regex:^T-[0-9]+_(nb_hour_under_0)$::_view[-1,${{n_lookback_days}},1], "
                        # f"regex:^T-[0-9]+_(nb_hour_over_0)$::_view[-1,${{n_lookback_days}},1], "
                        f"input_volume_lagged_view[-1,${{n_lookback_days}},1]"
                    f"]",
                    # f"concat["
                    f"input_temperature_forecast_view[-1,${{horizon}},24]",
                        # f"weather_forecast(apparent_temperature)_view[-1,${{horizon}},24], "
                        # f"weather_forecast(shortwave_radiation)_view[-1,${{horizon}},24], "
                        # f"regex:^T[0-9]+_(temp_min)$::_view[-1,${{horizon}},1], "
                        # f"regex:^T[0-9]+_(temp_max)$::_view[-1,${{horizon}},1], "
                        # f"regex:^T[0-9]+_(dj_0)$::_view[-1,${{horizon}},1], "
                        # f"regex:^T[0-9]+_(dj_0_sum)$::_view[-1,${{horizon}},1]"
                        # f"regex:^T[0-9]+_(nb_hour_under_0)$::_view[-1,${{horizon}},1], "
                        # f"regex:^T[0-9]+_(nb_hour_over_0)$::_view[-1,${{horizon}},1]"
                        # f"]",
                    f"target_volume_view[-1,${{horizon}},1]",
                    "volume_journalier_moyen",
                ],
            ],
        },
        "training_args": {
            "num_epochs": [1000],
            "batch_size": [64],
            "learning_rate": [0.005],
            "lstm_lr": [0.01,],
            "weight_decay": [1e-4],
            "dataType": [torch.bfloat16],
            "verbose": [False],
            "early_stopping_patience": [100],
            "early_stopping_min_delta": [0.000001],
            "early_stopping_min_epoch": [200],
        },
    },
)

new_exp = Experiment(
    comment="TSMixer avec bfloat16 !!Seulement Temperature!! H3et7",
    folder_path="models_testing/tsmixer_ext/run_oct2026_1",
    param_grid=newTester,
    experiment_function=exp_acer_data_module,
    exp_args_mapping={
        "model_ExpConstructor": Experiment.ExpArgConstructor(
            constructor = DirectConstructor,
            exp_args_mapping = {
                "constructor$&noSearch" : TSMixerExt_PositiveOutput,
                "args" : {
                    "config" : Experiment.ExpArgConstructor(
                        constructor = TSMixerExt_Config,
                        exp_args_mapping = {
                            "historical_time_dim" : "shared_params.lookback_days",
                            "historical_channel_dim" : "model_TSMixerExt.historical_channel_dim",
                            "future_time_dim" : "shared_params.horizon",
                            "future_channel_dim" : "model_TSMixerExt.future_channel_dim",
                            "mixing_hidden_dim" : "model_TSMixerExt.mixing_hidden_dim",
                            "k_mixing_layers" : "model_TSMixerExt.k_mixing_layers",
                            "output_channel_dim" : "shared_params.nb_predic_features",
                            "norm_placement" : "model_TSMixerExt.norm_placement",
                            "activation" : "model_TSMixerExt.activation",
                            "dropout_rate" : "model_TSMixerExt.dropout_rate",
                        }
                    ),
                }
            }
        ),
        "data_module": Experiment.ExpArgConstructor(
            constructor = ACERDataModule,
            exp_args_mapping = {
                "__$&UnpackDirect": "data_module_params",
                "n_lookback_days" : "shared_params.lookback_days",
                "horizon" : "shared_params.horizon"
            }
        ),
        "training_args" : Experiment.ExpArgConstructor(
            constructor = TrainingArgs,
            exp_args_mapping = {
                "__$&UnpackDirect": "training_args",
            }
        ),
        "exp_args": Experiment.ExpArgConstructor(
            constructor = Exp_Args,
            exp_args_mapping = {
                "path" : "__$&FolderPath",
                "model_base_folder$&noSearch" : "__$&ExpPath/saved_models/",
                "model_folder_division" : {
                    # "dType" : "training_args.dataType",
                    # "batch_size" : "training_args.batch_size",
                    "horizon" : "shared_params.horizon",
                    "lookback_days" : "shared_params.lookback_days"
                },
                "nb_trials$&noSearch" : 2,
                "exp_number" : "__$&RunIndex"
            }
        )
    }
)

# Le nombre de combinaisons
print(f"Nombre de combinaisons : {new_exp.get_combinations_count()}")

new_exp.save_to_yaml()

Nombre de combinaisons : 324


# RUN

In [1]:
import torch

# Activer les fonctionnalités de réglage de PyTorch pour les GPU AMD
torch.cuda.tunable.enable(True)
torch.cuda.tunable.record_untuned_enable(True)
print(torch.cuda.tunable.is_enabled())
print(torch.cuda.tunable.tuning_is_enabled())
print(torch.cuda.tunable.record_untuned_is_enabled())
torch.backends.cudnn.benchmark = False  # ignored on ROCm but sometimes copied

True
True
True


In [2]:
import os, yaml
from models.ParamGrid import Experiment
import utils.yaml.YAML_register_groups as YAMLGroups
YAMLGroups.register_torch()
from models.acer_conf_lstm import AcerConfLSTM, MLPBlock

# run_folder = "models_testing/tsmixer_ext/run_11_1_AppTemp"
# run_folder = "models_testing/LSTM/EXP_dropout_explorer"
# run_folder = "models_testing/tsmixer_ext/run_oct2026_1"
# run_folder = "models_testing/tsmixer_ext_more_exog/all_var/run_7_1_2"
run_folder = "models_testing/TSMixerExt_2TimeIngress/EXP_1_0"
yaml_file_path = run_folder + "/experiment.yaml"
result_path = os.path.join(run_folder, "results.yaml")

expToRun = Experiment.load_from_yaml(yaml_file_path)

# Obtenir le dernier numero d'experiment déjà présent dans le fichier de résultats.yaml pour éviter les écrasements
last_experiment_num = 0
if os.path.exists(result_path):
    print(f"Found existing results file at {result_path}. Checking for existing experiments to avoid overwriting...")
    with open(result_path, "r") as f:
        results = yaml.safe_load(f)
        if results is not None:
            experiment_nums = [int(key.split("_")[1]) for key in results.keys() if key.startswith("experiment_")]
            if experiment_nums:
                last_experiment_num = max(experiment_nums)+1
    try:
        del f, results, experiment_nums
    except NameError:
        pass

print(f"Found {expToRun.get_combinations_count()} configurations to run. Starting with experiment number {last_experiment_num}.")

best_exp = None
best_exp_val_loss = float("inf")
best_exp_WAPE = float("inf")

for i in range(last_experiment_num, expToRun.get_combinations_count()):
    print(f"=== Running experiment number {i} out of {expToRun.get_combinations_count() - 1}...")
    best_output = expToRun.run(index=i)

    if best_output["best_val_loss"] < best_exp_val_loss:
        best_exp = (i, best_output)
        best_exp_val_loss = best_output["best_val_loss"]
        best_exp_WAPE = best_output["history"]["val_wape"][best_output["best_val_epoch"]]

    print(f"== Bestst exp: {best_exp[0]} with val loss {best_exp_val_loss:.6f} and val WAPE {best_exp_WAPE:.6f}.")

# Sauvegarder le meilleur modèle et ses résultats
with open(os.path.join(run_folder, "best_experiment.yaml"), "w") as f:
    yaml.safe_dump({
        "best_experiment_index": best_exp[0],
        "best_val_loss": best_exp_val_loss,
        "best_val_WAPE": best_exp_WAPE,
        "best_model_path": best_exp[1]["trained_model_path"],
    }, f)

Found 72 configurations to run. Starting with experiment number 0.
=== Running experiment number 0 out of 71...
=== Starting experiment 0 ===
	Run 1/2 >>> 	Val loss: 0.357313 & Val loss orig: 5.883212 at epoch 205 out of 305.
	Run 2/2 >>> 	Val loss: 0.333300 & Val loss orig: 5.452555 at epoch 118 out of 299.
	Experiment 0 completed.
		Best val loss: 0.333300 at epoch 118 on run 2.	Val Loss:   0.333300	- Val Loss Orig:   5.452555	- Val WAPE: 59.36%	- Val RMSE%: 81.13%
== Bestst exp: 0 with val loss 0.333300 and val WAPE 59.361794.
=== Running experiment number 1 out of 71...
=== Starting experiment 1 ===
	Run 1/2 >>> 	Val loss: 0.323211 & Val loss orig: 5.287105 at epoch 292 out of 392.
	Run 2/2 >>> 	Val loss: 0.341707 & Val loss orig: 5.618005 at epoch 257 out of 357.
	Experiment 1 completed.
		Best val loss: 0.323211 at epoch 292 on run 1.	Val Loss:   0.323211	- Val Loss Orig:   5.287105	- Val WAPE: 56.42%	- Val RMSE%: 79.89%
== Bestst exp: 1 with val loss 0.323211 and val WAPE 56.424